In [213]:
# Importando bibliotecas
from faker import Faker
from datetime import datetime, timedelta
from PIL import Image
from io import BytesIO
from random import choice, randint, uniform, choices
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from unidecode import unidecode
from hashlib import sha256
from openai import OpenAI
import re

In [214]:
# Carregando variáveis de ambiente
load_dotenv()

# Acessando o BD principal e o anonimizado, e chave de api da hugging face
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_PRINC')
HUGGING_FACE = os.getenv('HUGGING_FACE')

# Criando engine
entrada = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

engine = create_engine(entrada)

# Carregando faker
f = Faker('pt_BR')

In [215]:
# Pega o último id das tabelas

def ultimos_ids(conn, tabela):
    result = conn.execute(text(f'SELECT ID FROM {tabela} ORDER BY ID DESC LIMIT 1'))
    return result.scalar()

with engine.connect() as conn:
    tabelas = ['PLANO', 'EMPRESA', 'MIDIA', 'TELEFONE_EMPRESA', 'EMPREGADO', 'SETOR_TURNO', 'MOTIVO', 'EMBALAGEM', 'DESCARTE']
    
    ultimos = {}
    for t in tabelas:
        ultimos[t] = ultimos_ids(conn, t)

In [216]:
def gerar_observacao(embalagem, motivo):
    prompt = f'Você é um operário que irá registrar uma pequena observação em um descarte de {embalagem} causados por {motivo}, em uma frase sem formatação, se o motivo for \'Outro\', invente. Seja conciso.'
    
    client = OpenAI (
        base_url='https://router.huggingface.co/v1',
        api_key=HUGGING_FACE
    )
    
    completion = client.chat.completions.create (
        model = 'moonshotai/Kimi-K2-Instruct-0905',
        messages = [
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0.2,
        max_tokens=30
    )
    
    return completion.choices[0].message.content

In [217]:
# Gerando informações de plano

planos = [
    {"nome": "Standard Mensal", "descricao": "Plano básico, ideal para pequenas indústrias", "preco": 149.99},
    {"nome": "Standard Anual", "descricao": "Plano básico, ideal para pequenas indústrias", "preco": 1439.90},
    {"nome": "Pro Mensal", "descricao": "Plano intermediário, ideal para indústrias em expansão", "preco": 199.99},
    {"nome": "Pro Anual", "descricao": "Plano intermediário, ideal para indústrias em expansão", "preco": 1919.90},
    {"nome": "Premium Mensal", "descricao": "Plano completo, ideal para grandes indústrias", "preco": 399.99},
    {"nome": "Premium Anual", "descricao": "Plano completo, ideal para grandes indústrias", "preco": 3839.90}
]

df_planos = pd.DataFrame(planos)

ultimo_id = (ultimos['PLANO'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_planos)))
df_planos.insert(0, 'id', ids)

display(df_planos)

,id,nome,descricao,preco
0,1,Standard Mensal,"Plano básico, ideal para pequenas indústrias",149.99
1,2,Standard Anual,"Plano básico, ideal para pequenas indústrias",1439.90
2,3,Pro Mensal,"Plano intermediário, ideal para indústrias em ...",199.99
3,4,Pro Anual,"Plano intermediário, ideal para indústrias em ...",1919.90
4,5,Premium Mensal,"Plano completo, ideal para grandes indústrias",399.99
5,6,Premium Anual,"Plano completo, ideal para grandes indústrias",3839.90


In [218]:
# Gerando informações de empresa

empresas = []

for i in range(40):
    empresa = f.company()
    empresa_email = re.sub('[./-]', ' ', empresa.lower()).strip().replace(' ', '_')
    dominios = ['@email.com.br', '@email.br', '@email.org.br', '@email.com']

    empresas.append({
        "nome": (empresa + ' ' + f.company_suffix()),
        "email": (empresa_email + str(randint(1,99)) + choice(dominios)),
        "cnpj": (f.cnpj()),
        "uf": (f.estado_sigla()),
        "cidade": (f.city()),
        "bairro": (f.neighborhood()),
        "cep": re.sub(r'\D', '', f.postcode()),
        "rua": (f.street_name()),
        "numero": (f.building_number()),
        "cod_plano": choice(df_planos['id'])
    })

df_empresas = pd.DataFrame(empresas)

ultimo_id = (ultimos['EMPRESA'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_empresas)))
df_empresas.insert(0, 'id', ids)

display(df_empresas)

,id,nome,email,cnpj,uf,cidade,bairro,cep,rua,numero,cod_plano
0,1,da Costa S.A.,da_costa77@email.org.br,49.735.082/0001-01,MA,da Costa Paulista,Aarão Reis,96669721,Condomínio Sousa,84,6
1,2,Aragão e Filhos,aragão84@email.com.br,62.980.534/0001-89,RO,Sampaio do Campo,Andiroba,07344126,Largo Anthony Pacheco,897,5
2,3,Nogueira Nogueira - ME - ME,nogueira_nogueira___me16@email.org.br,48.163.975/0001-67,RS,Cavalcante,Paraíso,52100591,Estação de da Cruz,9,2
3,4,Fernandes S.A.,fernandes28@email.com,75.861.423/0001-03,CE,Moreira,Funcionários,91313222,Rua Daniela Caldeira,68,1
4,5,Barros Ltda.,barros19@email.com.br,25.603.917/0001-01,BA,Duarte,Vista Do Sol,20992186,Rua Oliveira,50,6
5,6,Pereira S/A S/A,pereira_s_a95@email.org.br,51.867.042/0001-65,MA,Silveira de Pimenta,Mariano De Abreu,59197114,Vereda de Albuquerque,31,4
6,7,Montenegro S/A,montenegro65@email.com,93.485.170/0001-94,AC,Lima,Santo Antônio,38311661,Viaduto de Vasconcelos,64,6
7,8,da Conceição Ltda. S.A.,da_conceição_ltda71@email.org.br,15.978.320/0001-98,RO,Rocha,Vila São Gabriel,49922226,Residencial Marcelo Costa,16,2
8,9,Nogueira S.A. - ME,nogueira_s_a55@email.br,86.592.073/0001-35,CE,Costa,Beija Flor,86220163,Feira de Cassiano,5,2
9,10,Lima - EI,lima66@email.org.br,90.873.465/0001-40,MA,Dias de Novaes,Renascença,43344705,Travessa Alana Pinto,636,1


In [219]:
# Gerando informações de mídia

midias = []

for i in range(20):
    imagem = Image.new('RGB', (800, 600))
    memoria = BytesIO()
    imagem.save(memoria, format="jpeg")
    
    bytes_imagem = memoria.getvalue()
    memoria.close()
    
    midias.append({
        "caminho_arquivo": f.file_path(depth=2, category='image', extension='jpeg'),
        "arquivo": bytes_imagem,
        "horario": f.date_time_this_year()
    })

df_midias = pd.DataFrame(midias)

ultimo_id = (ultimos['MIDIA'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_midias)))
df_midias.insert(0, 'id', ids)

display(df_midias)

,id,caminho_arquivo,arquivo,horario
0,1,/illum/corporis/amet.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-07-15 17:36:38
1,2,/itaque/repellat/omnis.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-05-08 14:33:48
2,3,/error/atque/voluptate.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-04-10 03:57:15
3,4,/voluptates/vel/distinctio.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-01-13 07:00:22
4,5,/ut/atque/aliquid.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-06-04 04:22:46
5,6,/a/culpa/ipsum.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-02-27 04:27:39
6,7,/dolore/reprehenderit/blanditiis.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-07-16 09:35:08
7,8,/ullam/culpa/consectetur.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-02-17 23:21:04
8,9,/omnis/culpa/dolore.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-08-27 06:26:03
9,10,/molestias/neque/numquam.jpeg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2025-02-27 16:03:01


In [220]:
# Gerando informações de telefone

telefones = []

for i in range (50):
    telefones.append({
        "telefone": re.sub(r'\D', '', f.cellphone_number()),
        "cod_empresa": choice(df_empresas['id'])
    })

df_telefones = pd.DataFrame(telefones)

ultimo_id = (ultimos['TELEFONE_EMPRESA'] or 0) + 1
ids = range(ultimo_id, (ultimo_id + len(df_telefones)))
df_telefones.insert(0, 'id', ids)

display(df_telefones)

,id,telefone,cod_empresa
0,1,5562925769353,8
1,2,5560929513801,6
2,3,5528913291937,3
3,4,5595916752643,2
4,5,5564973793569,26
5,6,5549931301052,13
6,7,5569949592582,31
7,8,5518970332756,12
8,9,5543929574968,24
9,10,5578969790464,25


In [221]:
# Gerando informações de empregado

empregados = []
empregos = ["Operador de Produção", "Operador de Palletização", "Auxiliar de Almoxarifado", "Conferente de Materiais", "Separador de Carga", "Motorista de Empilhadeira", "Técnico de Manutenção", "Analista de Controle de Estoque", "Supervisor de Logística", "Gestor de Embalagens", "Coordenador de Produção"]
probabilidades = [30, 20, 15, 10, 8, 5, 4, 3, 3, 3, 1]

for i in range (50):
    nome = f.first_name()
    sobrenome = " " + f.last_name()
    email = unidecode(nome.lower().replace(' ', '_').replace('.', '') + sobrenome.lower().strip().replace(' ', '_').replace('.', ''))
    cargo = choices(empregos, weights=probabilidades, k=1)[0]
    
    if cargo in ['Analista de Controle de Estoque', 'Supervisor de Logistica', 'Gestor de Embalagens']:
        tipo = 'GESTOR'
    else:
        tipo = 'OPERARIO'
    
    empregados.append({
        "nome": nome,
        "sobrenome": sobrenome,
        "cpf": f.cpf(),
        "email": email + choice(dominios),
        "senha": sha256(f.password().encode()).hexdigest(),
        "telefone_empregado": re.sub(r'\D', '', f.cellphone_number()),
        "cargo": cargo,
        "tipo": tipo,
        "uf": (f.estado_sigla()),
        "cidade": (f.city()),
        "bairro": (f.neighborhood()),
        "cep": re.sub(r'\D', '', f.postcode()),
        "rua": (f.street_name()),
        "numero": (f.building_number()),
        "cod_empresa": choice(df_empresas['id'])
    })

df_empregados = pd.DataFrame(empregados)

ultimo_id = (ultimos['EMPREGADO'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_empregados)))
df_empregados.insert(0, 'id', ids)

display(df_empregados)

,id,nome,sobrenome,cpf,email,senha,telefone_empregado,cargo,tipo,uf,cidade,bairro,cep,rua,numero,cod_empresa
0,1,Marcos Vinicius,Nascimento,815.370.496-66,marcos_viniciusnascimento@email.com,d0ddaae7112e495e44e2aec19cebd3ebc76f3bbc9a4504...,5590904661802,Operador de Palletização,OPERARIO,SE,Pinto,Cardoso,90094276,Jardim Ana Julia Sousa,96,24
1,2,Otto,Vasconcelos,802.419.637-96,ottovasconcelos@email.org.br,dd94cbc50831c470226c8264313c47cd7f35ea9d1277ac...,5507978953862,Operador de Produção,OPERARIO,PE,Dias,Vila Madre Gertrudes 1ª Seção,64317263,Vale Luan Jesus,43,10
2,3,Gael,Marques,490.863.517-00,gaelmarques@email.com,33bb60f323ebf3185313f29e2907e8576d56868ca4bbca...,5554947438369,Operador de Palletização,OPERARIO,SP,da Conceição,Embaúbas,34103768,Parque Cassiano,4,7
3,4,Diogo,Brito,413.280.769-03,diogobrito@email.br,b38072ef082110b7ce2c659eb6d4853dd54ce84453dd1c...,5552923657962,Auxiliar de Almoxarifado,OPERARIO,TO,Fonseca do Norte,Vila Formosa,04243230,Jardim Azevedo,95,39
4,5,Henry,Cavalcante,615.827.340-62,henrycavalcante@email.com,64de4f3eb08d3e5b07e92cd60a57db31789bfa6bbd45ed...,5563981641764,Operador de Produção,OPERARIO,PI,Martins,São José,19173527,Distrito Sales,71,9
5,6,Ana Beatriz,Leão,730.249.158-50,ana_beatrizleao@email.com.br,81edd452f1c5b7e68b2fbd5fa947d03236fe0ab853fbd4...,5576955642863,Operador de Palletização,OPERARIO,RJ,Nunes de Farias,Nossa Senhora De Fátima,15871318,Setor Teixeira,72,22
6,7,Antonella,Lima,396.408.175-20,antonellalima@email.com,9f52edba6467c2a883111a8a1c027b26e7f00cbed6aeeb...,5509994528202,Operador de Produção,OPERARIO,AC,Porto,Centro,36621005,Trevo Camargo,90,21
7,8,Luan,Leão,961.540.273-70,luanleao@email.br,2771f23f5e28dccf5ba9559213e23913be400548d7076b...,5527961121893,Operador de Produção,OPERARIO,AC,Silveira,Coração De Jesus,48691355,Feira Caldeira,2,17
8,9,Liam,Vasconcelos,149.653.807-20,liamvasconcelos@email.br,a40613dc78e9b655fbbcc8da2d074c5be6f1fed3a217fa...,5534977809288,Operador de Produção,OPERARIO,PR,Jesus,Miramar,41105744,Trevo Câmara,65,6
9,10,Davi Miguel,Camargo,126.750.483-80,davi_miguelcamargo@email.com,c8ce95c71b97e1b490da5dbb626632e175c5bde6ad97ed...,55059938490399,Operador de Produção,OPERARIO,DF,Correia Alegre,Vila Maria,18522172,Viela Ester Machado,981,31


In [222]:
# Gerando setor e turno

setor_turno = []

for i in range(5):
    hora_inicio = f.time()
    while hora_inicio > '15:59:59':
        hora_inicio = f.time()
    
    datetime_inicio = datetime.strptime(hora_inicio, '%H:%M:%S')
    duracao = randint(5, 8)
    datetime_fim = datetime_inicio + timedelta(hours=duracao)
    
    hora_fim = datetime_fim.strftime('%H:%M:%S')
    
    setor_turno.append({
        "nome": f'Setor {i+1}',
        "hora_inicio": hora_inicio,
        "hora_fim": hora_fim
        })

df_setor_turno = pd.DataFrame(setor_turno)

ultimo_id = (ultimos['SETOR_TURNO'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_setor_turno)))
df_setor_turno.insert(0, 'id', ids)

display(df_setor_turno)

,id,nome,hora_inicio,hora_fim
0,1,Setor 1,03:59:01,10:59:01
1,2,Setor 2,10:42:30,17:42:30
2,3,Setor 3,12:32:01,18:32:01
3,4,Setor 4,04:09:38,09:09:38
4,5,Setor 5,05:10:45,11:10:45


In [223]:
# Gerando motivos

motivos = [
    {"nome": "Contaminação", "descricao": "Resíduos de produtos anteriores na embalagem"},
    {"nome": "Falha de Selagem", "descricao": "Vazamento de embalagem a vácuo"},
    {"nome": "Superaquecimento", "descricao": "Geração de calor excessiva pela ciclagem"},
    {"nome": "Desalinhamento", "descricao": "Adesivos/Impressão/Carimbagem errada"},
    {"nome": "Outro", "descricao": "Problema geral na embalagem"}
]

df_motivos = pd.DataFrame(motivos)

ultimo_id = (ultimos['MOTIVO'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_motivos)))
df_motivos.insert(0, 'id', ids)

display(df_motivos)

,id,nome,descricao
0,1,Contaminação,Resíduos de produtos anteriores na embalagem
1,2,Falha de Selagem,Vazamento de embalagem a vácuo
2,3,Superaquecimento,Geração de calor excessiva pela ciclagem
3,4,Desalinhamento,Adesivos/Impressão/Carimbagem errada
4,5,Outro,Problema geral na embalagem


In [224]:
# Gerando embalagem
embalagens = [
    {"nome": "Garrafa PET", "tipo": "Plástico", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Lata de Refrigerante", "tipo": "Metal", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Caixa de Papelão", "tipo": "Papelão", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Embalagem Plástica", "tipo": "Plástico PEAD", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)},
    {"nome": "Embalagem de Papel", "tipo": "Papel Reciclado", "fornecedor": f.company(), "valor_kg": round(uniform(0.0, 10.0), 2)}
]

df_embalagens = pd.DataFrame(embalagens)

ultimo_id = (ultimos['EMBALAGEM'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_embalagens)))
df_embalagens.insert(0, 'id', ids)

display(df_embalagens)

,id,nome,tipo,fornecedor,valor_kg
0,1,Garrafa PET,Plástico,Mendes,1.70
1,2,Lata de Refrigerante,Metal,da Conceição,1.92
2,3,Caixa de Papelão,Papelão,Aragão,0.95
3,4,Embalagem Plástica,Plástico PEAD,Moreira - EI,9.22
4,5,Embalagem de Papel,Papel Reciclado,Brito - EI,8.81


In [225]:
# Gerando informações de descarte

descartes = []

for i in range(120):
    descarte_kg = round(uniform(0.5, 10.0), 1)
    cod_emabalagem = choice(df_embalagens['id'])
    cod_motivo = choice(df_motivos['id'])
    
    valor_kg = df_embalagens.loc[df_embalagens['id'] == cod_emabalagem, 'valor_kg'].values[0]
    motivo = df_motivos.loc[df_motivos['id'] == cod_motivo, 'nome'].values[0]
    embalagem = df_embalagens.loc[df_embalagens['id'] == cod_emabalagem, 'nome'].values[0]
        
    descartes.append({
        "descarte_kg": descarte_kg,
        "custo_estimado": round((descarte_kg * valor_kg), 2),
        "horario": f.date_time_this_year(),
        "observacao": None,
        "cod_embalagem": cod_emabalagem,
        "cod_motivo": cod_motivo,
        "cod_empregado": choice(df_empregados['id']),
        "cod_midia": choice([choice(df_midias['id']), None])
    })

df_descartes = pd.DataFrame(descartes)

ultimo_id = (ultimos['DESCARTE'] or 0) + 1
ids = range(ultimo_id,  (ultimo_id + len(df_descartes)))
df_descartes.insert(0, 'id', ids)

display(df_descartes)

,id,descarte_kg,custo_estimado,horario,observacao,cod_embalagem,cod_motivo,cod_empregado,cod_midia
0,1,2.6,4.99,2025-05-03 01:46:49,None,2,3,5,NaN
1,2,3.2,3.04,2025-05-30 21:54:59,None,3,2,2,19.0
2,3,5.5,48.46,2025-09-23 19:53:16,None,5,4,46,10.0
3,4,8.1,7.69,2025-08-16 07:09:47,None,3,2,32,NaN
4,5,5.0,9.60,2025-07-15 01:24:49,None,2,3,41,1.0
...,...,...,...,...,...,...,...,...,...
115,116,4.9,8.33,2025-05-17 06:53:52,None,1,3,5,19.0
116,117,5.0,8.50,2025-01-15 02:19:35,None,1,5,40,NaN
117,118,7.3,67.31,2025-08-31 10:41:28,None,4,3,21,NaN
118,119,8.1,7.69,2025-03-30 07:18:31,None,3,2,30,NaN


In [226]:
# Gerando informações de relacionamento

armazena = []
trabalha_em = []

for i in range(15):
    armazena.append ({
        "cod_embalagem": choice(df_embalagens['id']),
        "cod_setor_turno": choice(df_setor_turno['id'])
    })

for i in range (50):
    trabalha_em.append({
        "cod_setor_turno": choice(df_setor_turno['id']),
        "cod_empregado": choice(df_empregados['id'])
    })

df_armazena = pd.DataFrame(armazena)
df_trabalha_em = pd.DataFrame(trabalha_em)

print('ARMAZENA')
display(df_armazena)

print('\nTRABALHA EM')
display(df_trabalha_em)


ARMAZENA


,cod_embalagem,cod_setor_turno
0,2,4
1,4,5
2,4,2
3,5,5
4,3,3
5,5,4
6,2,1
7,2,4
8,3,5
9,5,4



TRABALHA EM


,cod_setor_turno,cod_empregado
0,1,38
1,1,22
2,2,15
3,4,42
4,3,50
5,4,2
6,2,8
7,5,35
8,4,21
9,1,12


In [227]:
# Inserindo no banco de dados
df_planos.to_sql('plano', engine, if_exists='append', index=False)
df_empresas.to_sql('empresa', engine, if_exists='append', index=False)
df_midias.to_sql('midia', engine, if_exists='append', index=False)
df_telefones.to_sql('telefone_empresa', engine, if_exists='append', index=False)
df_empregados.to_sql('empregado', engine, if_exists='append', index=False)
df_setor_turno.to_sql('setor_turno', engine, if_exists='append', index=False)
df_motivos.to_sql('motivo', engine, if_exists='append', index=False)
df_embalagens.to_sql('embalagem', engine, if_exists='append', index=False)
df_descartes.to_sql('descarte', engine, if_exists='append', index=False)
df_armazena.to_sql('armazena', engine, if_exists='append', index=False)
df_trabalha_em.to_sql('trabalha_em', engine, if_exists='append', index=False)

50